## Agentic Systems, Introduction to Agents
### **Zahraa Salloum, 202674524**

**Business:** *Second Spark*, a fictional electronics repair and refurbishing workshop in Hamra, Beirut. The full profile is in `me/about_business.pdf` and the short version is in `me/business_summary.txt`.

**Agent:** *Watt*, the cheerful bench assistant of Second Spark. Watt answers questions about the workshop, collects leads and logs feedback and questions it could not answer.

**What this notebook covers:**
1. **Persona and instructions** kept as two separate labeled blocks, then assembled into one `system_prompt`.
2. **Router**: a small LLM call that classifies each message as `general_question`, `lead_collection` or `feedback`, with a keyword fallback if that call fails.
3. **Two tools**: `record_customer_interest(email, name, message)` and `record_feedback(question)`, both documented with MCP style schemas and both logging through print and a file.
4. **Agent loop**: user input, chat history and tools are passed to the model, tool calls are executed and their results are fed back.
5. **Reflection (bonus)**: the draft reply is checked against the instructions and revised if needed. The draft and the final reply are both logged.
6. **Gradio ChatInterface** to demo the bot.

In [1]:
# Install dependencies
!pip install -q "gradio>=5.0,<6" openai python-dotenv pypdf


[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# imports
import os
import re
import json
import time
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError, InternalServerError
from pypdf import PdfReader
import gradio as gr

c:\Users\zahra\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Model Setup**

The API key is read from `.env` so it never appears in the notebook. The code uses the `openai` client for the chat completions. If a `GEMINI_API_KEY` is found it points that same client at the OpenAI compatible endpoint of Gemini, otherwise it uses `OPENAI_API_KEY` with OpenAI directly. This way the rest of the notebook is the same for both providers.

In [3]:
load_dotenv(override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if GEMINI_API_KEY:
    # Gemini through its OpenAI compatible endpoint // same client, same code
    client = OpenAI(
        api_key=GEMINI_API_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )
    MODEL_NAME = os.getenv("MODEL_NAME", "gemini-3.5-flash-lite")
elif OPENAI_API_KEY:
    client = OpenAI(api_key=OPENAI_API_KEY)
    MODEL_NAME = os.getenv("MODEL_NAME", "gpt-4o-mini")
else:
    raise RuntimeError("No API key found. Add GEMINI_API_KEY or OPENAI_API_KEY to the .env file.")

print("Model:", MODEL_NAME)

Model: gemini-3.5-flash-lite


**Rate Limit Handling**

Every model call in this notebook goes through `call_model`. The free tier of the API has two limits: a few requests per minute and a small number of requests per day for each model. One customer message needs three or four calls (router, agent, tool round, reflection), so both limits are easy to hit. The helper handles the two cases differently:
1. If the API asks for a short wait, it waits and tries again.
2. If the daily quota of the model is finished, it switches to the next model in `MODEL_CHAIN` and continues.
3. If the server of the provider fails for a moment, it waits a few seconds and tries again.

Because the model can change during a run, the router and reflection log lines also show which model was used.

In [4]:
# models to fall back on when the daily free quota of one is finished // Gemini only
MODEL_CHAIN = [MODEL_NAME]
if GEMINI_API_KEY:
    MODEL_CHAIN += [m for m in ["gemini-3.7-flash", "gemini-3.5-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]
                    if m != MODEL_NAME]

MAX_RETRIES = 6
LONG_WAIT = 90  # seconds // a longer wait than this means the daily quota is finished
SERVER_RETRIES = 3  # extra tries when the provider has a temporary server error

def call_model(**kwargs):
    """One chat completion call that survives rate limits and temporary server errors."""
    global MODEL_NAME
    attempt = 0
    server_errors = 0
    while True:
        try:
            return client.chat.completions.create(model=MODEL_NAME, **kwargs)
        except InternalServerError as e:
            # temporary problem on the provider side // wait a little and try again
            server_errors += 1
            if server_errors > SERVER_RETRIES:
                raise
            print(f"[SERVER ERROR] {type(e).__name__}, trying again in {5 * server_errors}s")
            time.sleep(5 * server_errors)
        except RateLimitError as e:
            # the API says how long to wait // default to 20 seconds if it does not
            found = re.search(r"retryDelay\W+(\d+)s", str(e))
            wait = int(found.group(1)) + 1 if found else 20
            if wait > LONG_WAIT:
                position = MODEL_CHAIN.index(MODEL_NAME)
                if position + 1 >= len(MODEL_CHAIN):
                    raise
                MODEL_NAME = MODEL_CHAIN[position + 1]
                print(f"[RATE LIMIT] daily quota finished, switching to {MODEL_NAME}")
                continue
            attempt += 1
            if attempt >= MAX_RETRIES:
                raise
            print(f"[RATE LIMIT] waiting {wait}s before trying again")
            time.sleep(wait)

**Loading the Business Knowledge**

The agent answers from two files only: the PDF profile and the text summary. Both are read once and placed in the system prompt, they are small enough that no retrieval step is needed.

In [5]:
ME_DIR = "me"

# full business profile (PDF)
reader = PdfReader(os.path.join(ME_DIR, "about_business.pdf"))
about_business = "\n".join((page.extract_text() or "") for page in reader.pages).strip()

# short summary (TXT)
with open(os.path.join(ME_DIR, "business_summary.txt"), "r", encoding="utf-8") as f:
    business_summary = f.read().strip()

print(f"PDF profile: {len(about_business)} characters | Summary: {len(business_summary)} characters")

PDF profile: 2838 characters | Summary: 2429 characters


**Persona**

This block is only about who the agent is. It shapes the tone and the style and contains no rules.

In [6]:
# PERSONA (who the agent is)
PERSONA = """You are Watt, the cheerful bench assistant of Second Spark, an electronics repair and refurbishing workshop in Hamra, Beirut.
You talk like a friendly technician who loves bringing dead gadgets back to life: warm, practical and a little playful.
You like a small repair or electricity pun now and then (a spark, a short circuit, a recharge), but never more than one per reply.
You speak as part of the team, so you say "we" and "our workshop" when talking about Second Spark.
You keep replies short and clear, usually two to five sentences, and you explain technical things in plain words."""

**Instructions**

This block is the operational guardrails. It says what the agent must and must not do, whatever the tone is.

In [7]:
# INSTRUCTIONS (operational guardrails)
INSTRUCTIONS = """1. Answer ONLY from the business summary and the business profile given below. Never invent prices, services, opening hours, warranties, team members or policies that are not written there.
2. If the answer to a question is not in the business information, do not guess. Call the record_feedback tool with the question, then tell the customer honestly that you do not have that information yet and that the team has been notified.
3. When a customer shows interest in a repair, a refurbished device, a workshop seat, a trade in or a Business Care Plan, encourage them to leave their name and email so the team can follow up.
4. Never end a lead conversation without asking for an email. Once you have BOTH a name and an email, call the record_customer_interest tool with the email, the name and a short note about what they want. Do not call it with a made up or missing email. Use the email exactly as the customer typed it. If it looks incomplete (for example "lina at gmail"), ask them to type the full address and never complete or guess it yourself.
5. Business Care Plans have no public price. Never quote a number for them, collect the lead instead.
6. When a customer gives feedback, a complaint or a suggestion, call the record_feedback tool with their words, thank them and stay polite even if they are upset.
7. Never claim that something was recorded unless the tool was actually called and returned a success status.
8. Stay in character as Watt and stay on the topic of Second Spark. If asked about something unrelated, politely steer back to what the workshop can help with.
9. Never diagnose a device with certainty over chat. Suggest the free diagnostic instead."""

**Tool Functions**

Two tools are available to the model:
1. `record_customer_interest(email, name, message)` saves a lead.
2. `record_feedback(question)` saves feedback or a question the agent could not answer.

Both tools log through the same `push` helper, which prints the event and appends it to a JSON Lines file inside `logs/`. The print makes the call visible in the notebook and the file keeps a record that the business could read later.

In [8]:
LOG_DIR = "logs"
os.makedirs(LOG_DIR, exist_ok=True)

EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")

def push(event, payload):
    """Logging mechanism shared by the tools: prints the event and appends it to logs/<event>.jsonl"""
    record = {"time": datetime.now().isoformat(timespec="seconds"), **payload}
    print(f"[PUSH] {event}: {json.dumps(record, ensure_ascii=False)}")
    with open(os.path.join(LOG_DIR, f"{event}.jsonl"), "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return record

def record_customer_interest(email, name, message):
    """Saves a lead: who the customer is and what they are interested in."""
    if not EMAIL_RE.fullmatch((email or "").strip()):
        # the model gets this back and asks the customer for a proper email
        return {"status": "error", "reason": "A valid email address is required before the lead can be recorded."}
    push("leads", {"email": email.strip(), "name": name.strip(), "message": message.strip()})
    return {"status": "recorded", "detail": f"Lead saved for {name.strip()} <{email.strip()}>."}

def record_feedback(question):
    """Saves customer feedback or a question the agent could not answer."""
    push("feedback", {"question": question.strip()})
    return {"status": "recorded", "detail": "Feedback or unanswered question saved for the team."}

TOOL_FUNCTIONS = {
    "record_customer_interest": record_customer_interest,
    "record_feedback": record_feedback,
}

**Tool Schemas (MCP Style)**

Each tool is described the way an MCP server lists its tools: a `name`, a short `description`, an `inputSchema` for the parameters and an `outputSchema` for what comes back. The chat completions API expects a slightly different wrapper, so `to_openai_tools` converts the same definitions instead of writing them twice.

`record_customer_interest`
1. Parameters: `email` (string, required), `name` (string, required), `message` (string, required).
2. Expected output: `{"status": "recorded", "detail": "..."}` on success, or `{"status": "error", "reason": "..."}` if the email is not valid.

`record_feedback`
1. Parameters: `question` (string, required).
2. Expected output: `{"status": "recorded", "detail": "..."}`.

In [9]:
MCP_TOOLS = [
    {
        "name": "record_customer_interest",
        "description": "Save a lead once the customer has given BOTH a name and an email.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "email": {"type": "string", "description": "The email address of the customer."},
                "name": {"type": "string", "description": "The name of the customer."},
                "message": {"type": "string", "description": "Short note on what the customer is interested in."},
            },
            "required": ["email", "name", "message"],
        },
        "outputSchema": {
            "type": "object",
            "properties": {
                "status": {"type": "string", "enum": ["recorded", "error"]},
                "detail": {"type": "string"},
                "reason": {"type": "string"},
            },
            "required": ["status"],
        },
    },
    {
        "name": "record_feedback",
        "description": "Log customer feedback, a complaint, or a question that can not be answered from the business information.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "question": {"type": "string", "description": "The feedback or the unanswered question, in the words of the customer."},
            },
            "required": ["question"],
        },
        "outputSchema": {
            "type": "object",
            "properties": {
                "status": {"type": "string", "enum": ["recorded"]},
                "detail": {"type": "string"},
            },
            "required": ["status"],
        },
    },
]

def to_openai_tools(names):
    """Converts the MCP style definitions into the format the chat completions API expects."""
    return [
        {"type": "function",
         "function": {"name": t["name"], "description": t["description"], "parameters": t["inputSchema"]}}
        for t in MCP_TOOLS if t["name"] in names
    ]

**Router**

Before the agent does anything, the message is classified into one of three intents:
1. `general_question`: the customer asks about the business.
2. `lead_collection`: the customer wants to book, buy, get a quote or is sharing contact details.
3. `feedback`: the customer gives feedback, a complaint or a suggestion.

The router is a small separate LLM call that returns JSON. It also sees the last few turns, because a message like "Sara, sara@mail.com" only makes sense as a lead if we know the agent just asked for it. If the call fails or returns something unexpected, a simple keyword heuristic takes over so the chat never breaks.

Each intent then goes down its own path in `ROUTES`: a different set of tools and a different extra instruction. The feedback route forces a tool call, so feedback is always logged and does not depend on the model remembering to do it.

In [10]:
INTENTS = ["general_question", "lead_collection", "feedback"]

ROUTER_PROMPT = """You are a message router for Second Spark, an electronics repair workshop. Classify the LATEST customer message into exactly one intent:

general_question: asks about the business (services, prices, hours, team, warranty, location) or is small talk.
lead_collection: wants to book a repair, buy a device, join a workshop, get a quote or be contacted, OR is giving a name or email.
feedback: gives feedback, a complaint, praise or a suggestion about the business.

Use the recent conversation only as context. Reply with JSON only, like {"intent": "general_question"}."""

LEAD_WORDS = ["book", "quote", "sign up", "register", "reserve", "contact me", "call me", "interested in",
              "appointment", "my email", "my name is", "i want to buy", "i would like to buy"]
FEEDBACK_WORDS = ["feedback", "complain", "complaint", "disappointed", "suggest", "bad experience",
                  "terrible", "unhappy", "not happy", "great job", "review"]

def heuristic_route(message):
    """Keyword fallback, used only if the LLM router fails."""
    text = message.lower()
    if EMAIL_RE.search(text) or any(w in text for w in LEAD_WORDS):
        return "lead_collection"
    if any(w in text for w in FEEDBACK_WORDS):
        return "feedback"
    return "general_question"

def route(message, history=None):
    """Returns (intent, source) where source says whether the LLM or the fallback decided."""
    recent = (history or [])[-4:]
    context = "\n".join(f"{m['role']}: {m['content']}" for m in recent) or "(no previous messages)"
    try:
        response = call_model(
            messages=[
                {"role": "system", "content": ROUTER_PROMPT},
                {"role": "user", "content": f"Recent conversation:\n{context}\n\nLATEST customer message:\n{message}"},
            ],
            response_format={"type": "json_object"},
        )
        raw = response.choices[0].message.content or ""
        intent = json.loads(re.search(r"\{.*\}", raw, re.DOTALL).group(0)).get("intent")
        if intent in INTENTS:
            return intent, "llm"
    except Exception as e:
        print(f"[ROUTER] LLM router failed ({type(e).__name__}), using keyword fallback")
    return heuristic_route(message), "heuristic"

# each intent gets its own tools and its own extra instruction
ROUTES = {
    "general_question": {
        "tools": ["record_feedback"],
        "force_tool": False,
        "note": "The customer is asking a general question. Answer it from the business information. "
                "If the answer is not there, call record_feedback with the question. "
                "If they sound interested in a service, invite them to leave a name and email.",
    },
    "lead_collection": {
        "tools": ["record_customer_interest", "record_feedback"],
        "force_tool": False,
        "note": "The customer is a potential lead. Find out what they need and get their name and email. "
                "If either one is missing, ask for it. Once you have both, call record_customer_interest.",
    },
    "feedback": {
        "tools": ["record_feedback"],
        "force_tool": True,
        "note": "The customer is giving feedback, a complaint or a suggestion. Log it with record_feedback, "
                "then thank them and respond with care.",
    },
}

**System Prompt**

Here the persona and the instructions are assembled into the `system_prompt`, together with the business summary and the PDF content. The prompt keeps the agent in character, limits it to the business information, tells it to log unknown questions through a tool and to encourage leads to leave their contact details. The route note is added at the end for the intent that the router picked.

In [11]:
system_prompt = f"""## PERSONA
{PERSONA}

## INSTRUCTIONS
{INSTRUCTIONS}

## BUSINESS SUMMARY
{business_summary}

## BUSINESS PROFILE (from about_business.pdf)
{about_business}"""

def build_system_prompt(intent):
    return system_prompt + f"\n\n## CURRENT ROUTE: {intent}\n{ROUTES[intent]['note']}"

print(system_prompt[:900] + "\n...")

## PERSONA
You are Watt, the cheerful bench assistant of Second Spark, an electronics repair and refurbishing workshop in Hamra, Beirut.
You talk like a friendly technician who loves bringing dead gadgets back to life: warm, practical and a little playful.
You like a small repair or electricity pun now and then (a spark, a short circuit, a recharge), but never more than one per reply.
You speak as part of the team, so you say "we" and "our workshop" when talking about Second Spark.
You keep replies short and clear, usually two to five sentences, and you explain technical things in plain words.

## INSTRUCTIONS
1. Answer ONLY from the business summary and the business profile given below. Never invent prices, services, opening hours, warranties, team members or policies that are not written there.
2. If the answer to a question is not in the business information, do not guess. Call the re
...


**Agent Interaction**

`run_agent` passes the user input, the chat history (short term memory) and the tools of the chosen route to the model. If the model asks for a tool, the tool is executed and its result is sent back so the model can write the final reply. If no tool is requested, the generated text is returned directly. The loop is capped at a few rounds so it can never run forever.

There is also one guard in code: before `record_customer_interest` runs, it checks that the email really appears in what the customer typed. If the model tries to save an address it guessed, for example turning "lina at gmail" into lina@gmail.com, the tool returns an error and the model has to ask the customer for the full email.

In [12]:
MAX_TOOL_ROUNDS = 4

def handle_tool_call(call, customer_text=""):
    """Runs one tool call requested by the model and returns the result as a dict."""
    name = call.function.name
    try:
        args = json.loads(call.function.arguments or "{}")
        email = str(args.get("email", "")).strip().lower()
        if name == "record_customer_interest" and email and email not in customer_text.lower():
            # guard // the email must be one the customer actually typed, the model can not guess it
            result = {"status": "error", "reason": "The customer never typed this email. Ask them to type their full email address."}
        else:
            result = TOOL_FUNCTIONS[name](**args)
    except KeyError:
        result = {"status": "error", "reason": f"Unknown tool {name}."}
        args = {}
    except (TypeError, json.JSONDecodeError) as e:
        result = {"status": "error", "reason": f"Bad arguments: {e}"}
        args = {}
    return {"tool": name, "arguments": args, "result": result}

def run_agent(message, history, intent):
    """Returns (draft reply, list of tool calls that were made)."""
    config = ROUTES[intent]
    tools = to_openai_tools(config["tools"])
    messages = [{"role": "system", "content": build_system_prompt(intent)}] + history + [{"role": "user", "content": message}]
    tool_events = []
    # everything the customer typed so far // used to check that a saved email is real
    customer_text = "\n".join(m["content"] for m in history if m["role"] == "user") + "\n" + message

    for step in range(MAX_TOOL_ROUNDS):
        kwargs = dict(messages=messages, tools=tools)
        if config["force_tool"] and step == 0:
            # feedback route // the first step must be a tool call
            kwargs["tool_choice"] = "required"
        response = call_model(**kwargs)
        reply = response.choices[0].message

        if not reply.tool_calls:
            return (reply.content or "").strip(), tool_events

        # the model asked for a tool // run it and send the result back
        messages.append(reply)
        for call in reply.tool_calls:
            event = handle_tool_call(call, customer_text)
            tool_events.append(event)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(event["result"])})

    return "Sorry, something got tangled in my wires. Could you say that again?", tool_events

**Reflection Step (Bonus)**

Before the reply is sent, a second model call acts as a reviewer. It receives the instructions, the business information, the recent conversation, the customer message, the draft and the list of tools that were actually called. The conversation matters: without it the reviewer can not know that the customer already gave a name or an email in an earlier turn. It checks things like: did the draft invent a price, did it forget to ask for an email, did it say something was recorded when no tool ran. It answers in JSON with a verdict and, if needed, a revised reply.

Both the original draft and the final reply are printed and saved to `logs/reflection.jsonl`, so the self correction is visible. If the reflection call itself fails, the draft is sent as it is.

In [13]:
REFLECTION_PROMPT = """You are a strict quality reviewer for the customer assistant of Second Spark. You get the instructions the assistant must follow, the business information, the recent conversation, the latest customer message, the tools the assistant actually called and its DRAFT reply.

Check the draft against the instructions. In particular:
1. Does it state any price, service, hour, warranty or policy that is NOT in the business information?
2. If the customer is a lead and no lead was recorded yet, does it ask for the missing name or email?
3. Does it claim something was recorded or passed to the team when no tool call with status "recorded" is listed?
4. Does it stay in character as Watt, short and friendly?

Use the recent conversation as context: anything the customer already said there (a name, an email, what they want) counts as known.
Only set "passes" to false for a clear break of the instructions. Do not revise a draft just to change its style or wording.

Reply with JSON only:
{"passes": true or false, "issues": ["short description of each problem"], "revised_reply": "the corrected reply, or an empty string if the draft passes"}

If you revise, keep the voice of the draft and change only what is needed to fix the issues."""

reflection_log = []

def reflect(message, draft, tool_events, intent, history=None):
    """Checks the draft against the instructions and returns the final reply."""
    recent = (history or [])[-6:]
    context = "\n".join(f"{m['role']}: {m['content']}" for m in recent) or "(no previous messages)"
    review_input = (
        f"## INSTRUCTIONS\n{INSTRUCTIONS}\n\n"
        f"## BUSINESS INFORMATION\n{business_summary}\n\n{about_business}\n\n"
        f"## ROUTE\n{intent}\n\n"
        f"## TOOLS ACTUALLY CALLED\n{json.dumps(tool_events, ensure_ascii=False) if tool_events else 'none'}\n\n"
        f"## RECENT CONVERSATION\n{context}\n\n"
        f"## LATEST CUSTOMER MESSAGE\n{message}\n\n"
        f"## DRAFT REPLY\n{draft}"
    )
    final, issues, revised = draft, [], False
    try:
        response = call_model(
            messages=[
                {"role": "system", "content": REFLECTION_PROMPT},
                {"role": "user", "content": review_input},
            ],
            response_format={"type": "json_object"},
        )
        raw = response.choices[0].message.content or ""
        review = json.loads(re.search(r"\{.*\}", raw, re.DOTALL).group(0))
        issues = review.get("issues") or []
        revised_reply = (review.get("revised_reply") or "").strip()
        if review.get("passes") is False and revised_reply:
            final, revised = revised_reply, True
    except Exception as e:
        issues = [f"reflection skipped ({type(e).__name__})"]

    entry = push("reflection", {"intent": intent, "model": MODEL_NAME, "user": message, "draft": draft,
                                "revised": revised, "issues": issues, "final": final})
    reflection_log.append(entry)
    return final

**Putting It Together**

`chat` is the function Gradio calls for every message: route, run the agent on that route, reflect, reply.

In [14]:
def chat(message, history):
    # keep only role and content from the Gradio history // this is the short term memory
    history = [{"role": m["role"], "content": m["content"]} for m in (history or [])
               if m.get("role") in ("user", "assistant") and isinstance(m.get("content"), str)]

    intent, source = route(message, history)
    print(f"[ROUTER] {intent} (decided by {source}) | model: {MODEL_NAME}")

    draft, tool_events = run_agent(message, history, intent)
    return reflect(message, draft, tool_events, intent, history)

**Demonstration: Router**

A few messages to show that each one is sent down the right path.

In [15]:
router_tests = [
    "What are your opening hours?",
    "How much is a phone screen replacement?",
    "I want to book a seat in the Saturday workshop",
    "Can I get a quote for a care plan for my office of 12 people?",
    "My name is Lina and my email is lina@example.com",
    "My laptop came back with a scratch on the lid, I am really disappointed",
    "You should open on Sundays too",
]
for text in router_tests:
    intent, source = route(text)
    print(f"{intent:<17} ({source})  <-  {text}")

general_question  (llm)  <-  What are your opening hours?
general_question  (llm)  <-  How much is a phone screen replacement?
lead_collection   (llm)  <-  I want to book a seat in the Saturday workshop
lead_collection   (llm)  <-  Can I get a quote for a care plan for my office of 12 people?
lead_collection   (llm)  <-  My name is Lina and my email is lina@example.com
feedback          (llm)  <-  My laptop came back with a scratch on the lid, I am really disappointed
feedback          (llm)  <-  You should open on Sundays too


**Demonstration: Full Conversations**

The helper below plays a conversation turn by turn and keeps the history between turns, the same way Gradio does. The `[ROUTER]` and `[PUSH]` lines show the route that was picked, the tool calls and the reflection log.

In [16]:
def play(turns):
    """Runs a scripted conversation and prints it, keeping the history between turns."""
    history = []
    for text in turns:
        print(f"CUSTOMER: {text}")
        answer = chat(text, history)
        print(f"WATT: {answer}\n{'=' * 70}")
        history += [{"role": "user", "content": text}, {"role": "assistant", "content": answer}]

In [17]:
# general questions answered from the summary and the PDF
play([
    "Hi! What does Second Spark do?",
    "How much does a laptop battery replacement cost and is there a warranty?",
    "Who teaches the workshops?",
])

CUSTOMER: Hi! What does Second Spark do?
[ROUTER] general_question (decided by llm) | model: gemini-3.5-flash-lite
[PUSH] reflection: {"time": "2026-10-14T19:43:58", "intent": "general_question", "model": "gemini-3.5-flash-lite", "user": "Hi! What does Second Spark do?", "draft": "Hello! We at Second Spark are an electronics repair and refurbishing workshop on Makdessi Street in Hamra, Beirut. Our mission is to keep working gadgets out of the landfill by fixing phones, laptops, tablets, game consoles, and small audio gear—giving your tech a bright new spark!", "revised": false, "issues": [], "final": "Hello! We at Second Spark are an electronics repair and refurbishing workshop on Makdessi Street in Hamra, Beirut. Our mission is to keep working gadgets out of the landfill by fixing phones, laptops, tablets, game consoles, and small audio gear—giving your tech a bright new spark!"}
WATT: Hello! We at Second Spark are an electronics repair and refurbishing workshop on Makdessi Street in 

In [18]:
# a question that is NOT in the business information // should call record_feedback
play(["Do you offer data recovery from a dead hard drive?"])

CUSTOMER: Do you offer data recovery from a dead hard drive?
[ROUTER] general_question (decided by llm) | model: gemini-3.5-flash-lite
[PUSH] feedback: {"time": "2026-10-14T19:44:07", "question": "Do you offer data recovery from a dead hard drive?"}
[PUSH] reflection: {"time": "2026-10-14T19:44:09", "intent": "general_question", "model": "gemini-3.5-flash-lite", "user": "Do you offer data recovery from a dead hard drive?", "draft": "I am sorry, but I do not have that information yet, and I have notified the team about your question! We always like to recharge our knowledge when new questions spark up, so someone will follow up with you soon.", "revised": false, "issues": [], "final": "I am sorry, but I do not have that information yet, and I have notified the team about your question! We always like to recharge our knowledge when new questions spark up, so someone will follow up with you soon."}
WATT: I am sorry, but I do not have that information yet, and I have notified the team abou

In [19]:
# lead collection over several turns // should end with record_customer_interest
play([
    "I run a small school and I want a Business Care Plan for our 30 laptops. How much is it?",
    "Sure, I am Nour Aziz",
    "nour.aziz@example.com",
])

CUSTOMER: I run a small school and I want a Business Care Plan for our 30 laptops. How much is it?
[ROUTER] general_question (decided by llm) | model: gemini-3.5-flash-lite
[PUSH] reflection: {"time": "2026-10-14T19:44:12", "intent": "general_question", "model": "gemini-3.5-flash-lite", "user": "I run a small school and I want a Business Care Plan for our 30 laptops. How much is it?", "draft": "Hello there! We love helping schools keep their tech humming along without a short circuit in their budget. \n\nOur Business Care Plans for schools and small offices are tailored specifically to your needs and are quoted per customer after a short chat, so we don't have a fixed public price list for them. \n\nTo help us get things charged up and give you the right quote, could you please share your name and email address so our team can follow up with you?", "revised": false, "issues": [], "final": "Hello there! We love helping schools keep their tech humming along without a short circuit in the

In [20]:
# feedback // the route forces record_feedback
play(["I waited two days longer than promised for my phone repair, that was not okay."])

CUSTOMER: I waited two days longer than promised for my phone repair, that was not okay.
[ROUTER] feedback (decided by llm) | model: gemini-3.5-flash-lite
[PUSH] feedback: {"time": "2026-10-14T19:44:20", "question": "I waited two days longer than promised for my phone repair, that was not okay."}
[RATE LIMIT] waiting 38s before trying again
[RATE LIMIT] waiting 57s before trying again
[PUSH] reflection: {"time": "2026-10-14T19:46:02", "intent": "feedback", "model": "gemini-3.5-flash-lite", "user": "I waited two days longer than promised for my phone repair, that was not okay.", "draft": "We are so sorry for the delay with your phone repair, and we completely understand why that was frustrating for you. I have shared your feedback directly with our team so we can make sure our turnaround times stay sharp and reliable. Thank you for letting us know, and we appreciate your patience while we get your device back into shape!", "revised": false, "issues": [], "final": "We are so sorry for th

**Demonstration: Reflection Log**

Every reply above went through the reflection step. This prints the draft next to the final reply for each one, and marks the ones that were revised.

In [21]:
for i, entry in enumerate(reflection_log, 1):
    print(f"#{i} [{entry['intent']}] model: {entry['model']} | revised: {entry['revised']} | issues: {entry['issues']}")
    print(f"   DRAFT: {entry['draft']}")
    if entry["revised"]:
        print(f"   FINAL: {entry['final']}")
    print()

#1 [general_question] model: gemini-3.5-flash-lite | revised: False | issues: []
   DRAFT: Hello! We at Second Spark are an electronics repair and refurbishing workshop on Makdessi Street in Hamra, Beirut. Our mission is to keep working gadgets out of the landfill by fixing phones, laptops, tablets, game consoles, and small audio gear—giving your tech a bright new spark!

#2 [general_question] model: gemini-3.5-flash-lite | revised: False | issues: []
   DRAFT: A laptop battery replacement at our workshop starts from 40 USD, and all our repairs come with a solid 90-day warranty on parts and labor! We also provide a free diagnostic within 24 hours so you get a fixed price before any work begins. 

If you are thinking about bringing your laptop in, could you please share your name and email so our team can follow up with you?

#3 [general_question] model: gemini-3.5-flash-lite | revised: False | issues: []
   DRAFT: Jad Saliba, our fantastic workshop instructor, teaches our Saturday Fix 

**Demonstration: Reflection Catching a Bad Draft**

The models passed every draft in the run above, so this cell hands the reviewer a draft that breaks two instructions on purpose. It invents a monthly price for a Business Care Plan and it claims the details of the customer were saved when no tool was called. The reviewer should catch both problems and send back a corrected reply.


In [22]:
# a deliberately bad draft // invents a care plan price and claims a lead was saved without any tool call
test_message = "How much is a Business Care Plan for my office?"
bad_draft = ("Our Business Care Plan costs 49 USD per month for up to 10 devices! "
             "I have saved your details and Karim will call you tomorrow.")

final = reflect(test_message, bad_draft, tool_events=[], intent="lead_collection")
print(f"\nDRAFT: {bad_draft}\nFINAL: {final}")


[PUSH] reflection: {"time": "2026-10-14T19:46:03", "intent": "lead_collection", "model": "gemini-3.5-flash-lite", "user": "How much is a Business Care Plan for my office?", "draft": "Our Business Care Plan costs 49 USD per month for up to 10 devices! I have saved your details and Karim will call you tomorrow.", "revised": true, "issues": ["Quoted a price for Business Care Plans which have no public price and are quoted per customer.", "Claimed details were saved and someone would call when no tool was called and no name or email was collected."], "final": "Hi! I'm Watt from Second Spark. Business Care Plans are custom-quoted per customer after a short chat so there is no fixed price list for them. May I have your name and email so Karim from our team can get in touch with you about a plan for your office?"}

DRAFT: Our Business Care Plan costs 49 USD per month for up to 10 devices! I have saved your details and Karim will call you tomorrow.
FINAL: Hi! I'm Watt from Second Spark. Busine

**Gradio Interface**

In [23]:
demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    title="\u26A1 Second Spark, chat with Watt",
    description="Ask about repairs, refurbished devices, workshops and care plans. Fix it, do not bin it.",
    examples=[
        "What do you repair and how much does a screen replacement cost?",
        "I want to join the Saturday workshop",
        "Do you offer data recovery?",
        "I have some feedback about my last repair",
    ],
    theme=gr.themes.Soft(primary_hue="orange", secondary_hue="amber"),
)

**Launch the App**

In [24]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
